# Text clip finetune example


In [ ]:
import sys
import os
if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

import torch
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase
from utils.color.tonemapping.agx_looks import AgXPunchyLook

In [ ]:
torch_precision = torch.float32
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
lr = 0.05
n_iter = 500

In [ ]:
global_seed = 3 # Can be None

In [ ]:
import open_clip
from utils.train import train_with_criterion
from torchvision.transforms.v2 import RandomChoice, RandomPerspective, RandomResizedCrop, RandomHorizontalFlip, GaussianBlur, Identity, Transform
from losses.clip import CLIPDirectionalCosineSimilarity
from utils.model.model_utils import create_clip_model_and_tokenizer

clip_model_name = "ViT-B-16-SigLIP-512"
clip_pretrained = "webli"

from examples.example_scenes import SpringScene, SciFiRobotScene, CarScene, BlenderManScene, HouseScene, DinoScene, FlowerPotScene, RedCarScene, EinarSmallDomeScene, SpringPortraitSmallDomeScene
scene = SciFiRobotScene(device=device)
color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())

models_to_use = [
    None,                                           # Original pre-trained model
    "siglip_blend-training-data_64-output-dim.pt"   # Fine-tuned model (auto-downloaded from Hugging Face if needed)
]
for fine_tune in models_to_use:

    model, tokenizer, preprocess_eval = create_clip_model_and_tokenizer(
        clip_model_name,
        device=device,
        pretrained=clip_pretrained,
        fine_tune=fine_tune,
    )
    model.eval()
    print("Model loaded! :)")
    
    prompts = [
        # lighting description distribution
        ("boring, ugly, even", "stunningly beautiful lighting")
    ]

    for initial_prompt, target_prompt in prompts:
    
        criterion = CLIPDirectionalCosineSimilarity(initial_prompt, target_prompt, scene.get_combined_image(color_space_converter).permute(2, 1, 0), model, tokenizer, device=device, preprocess=preprocess_eval, always_prenormalize_vectors=True)
        title_prefix = "ImageTextFineTuning Comparison"

        size = model.visual.preprocess_cfg["size"] or (224, 224)
        
        train_with_criterion(
            scene,
            lr, n_iter, criterion,
            starting_multiplier_std=(0.3, 0.3, 0.3),
            output_subdirectory_name="text_clip_finetune_example",
            n_results=4,
            torch_precision=torch_precision,
            render_color_space_converter=color_space_converter,
            require_physically_plausible_multipliers=True,
            title_prefix=title_prefix + ("Fine-Tuned Model" if fine_tune else "Original Model"),
            device=device,
            save_every=25,
            model_name=clip_model_name,
            pretrained_source=fine_tune,
            seed=global_seed,
            show_images_after_augmentation=False
        )